# Imports

In [2]:
# Pytorch
import torch
from torch import nn

# BART model
from transformers import BartConfig, BartForConditionalGeneration
from transformers import AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset, load_metric

# Monitor Training
import wandb

# Saving Files with JSON
import json

# File path concatenation
from os.path import join
data_path = join('..', '..', 'data')

ModuleNotFoundError: No module named 'transformers.utils'

In [3]:
from Bio.Seq import Seq

In [8]:
import numpy as np
import torch
import torch.nn.functional as F
from ..Baseline_Models.mfc_harmonizer.c_50.mfc_harmonizer import Harmonizer

# Translate codons to amino acids
from Bio.Seq import Seq


def split_codons(example):
    """Split 'codon_seq' values by ' ' character"""
    return {'codon_seq': example['codon_seq'].upper().split(" ")}

def get_tokenize_codons(tokenizer):
    def tokenize(example):
        seqcodons = example['codon_seq']
        return tokenizer(seqcodons)
    return tokenize

def mfc_metric(tokenizer, path=None):
    if path is None:
        print("Path to MFC training data not provided!")
        exit()

    # Load train data from CSV and convert JSON strings back to lists
    train_df = pd.read_csv(path)
    train_df["median"] = train_df["median"].apply(lambda x: json.loads(x) if pd.notnull(x) else x)

    mfc_model = Harmonizer(train_df['amino_acid_seq'], train_df['seq'], verbose=True)

    def calc_mfc(masked_labels):
        total_dna = ''.join(tokenizer.decode(masked_labels)).replace(' ','')
        total_aa = Seq(total_dna).translate()
        
        predictions = mfc_model.predict_sequence(test_df['amino_acid_seq'], verbose=True)
        accuracies = mfc_model.calculate_accuracies(predictions, test_df['seq'], verbose=True)

        acc = mfc_model.calculate_accuracies(total_dna)  # TODO: finish fixing
        return acc

    return calc_mfc, mfc_model

def get_metrics(tokenizer, training_path=None, special_token_th=31):

    compute_mfc, mfc_model = mfc_metric(tokenizer, training_path)
    def compute_perplexity(predictions, labels, mask):
        predictions = predictions.transpose(2,1)
        loss = F.cross_entropy(predictions, labels, ignore_index=-100)

        perplexity = torch.exp(loss)
        return perplexity

    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions = torch.tensor(np.argmax(logits[0], axis=-1))
        labels = torch.tensor(labels)
        masked_index = torch.ne(labels, -100)
        special_token_index = torch.gt(labels, special_token_th)

        mask = torch.logical_and(masked_index, special_token_index)
        acc = torch.tensor(predictions == labels, dtype=torch.float32)
        acc = torch.masked_select(acc,mask)
        acc_mean = torch.mean(acc)

        mfc_score = compute_mfc(torch.masked_select(labels,mask))
        pp = compute_perplexity(torch.tensor(logits[0]), labels, mask)

        return {"acc": acc_mean, "mfc": mfc_score, "pp":pp, "acc_diff":(acc_mean - mfc_score)} 
    return compute_metrics

ImportError: attempted relative import with no known parent package